# Create Alcohol Change UK Awards

**Alcohol Research UK / Alcohol Change UK** (funder_id `4320319998` — the richer
legacy record of the pair w/ F4320327606, flagged in the registry memo; IAMHRF
expanded, priority `317`). alcoholchange.org.uk funded-projects listing across 5
schemes. 56 entries; org-level (no PI at source), institution 98%, year 75%
(14 undated at source); no amounts (§6.7).

Parquet: `s3://openalex-ingest/awards/alcohol_change_uk/alcohol_change_uk_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.alcohol_change_uk_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/alcohol_change_uk/alcohol_change_uk_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.alcohol_change_uk_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.alcohol_change_uk_raw;

## Step 2: Create ACUK Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.alcohol_change_uk_awards
USING delta
AS
WITH
acuk_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320319998  -- Alcohol Research UK (legacy/richer of the pair)
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'alcohol_change_uk' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.year_awarded AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.institution IS NOT NULL THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United Kingdom' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.alcohol_change_uk_raw g
    CROSS JOIN acuk_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'alcohol_change_uk' AND priority = 317;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    317 as priority
FROM openalex.awards.alcohol_change_uk_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(display_name) has_title, COUNT(lead_investigator.affiliation.name) has_inst, COUNT(start_year) has_year
FROM openalex.awards.alcohol_change_uk_awards;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'alcohol_change_uk' AND priority = 317;